In [ ]:
import sqlite3
import os
from pathlib import Path
import uuid

def write_file_to_database(db_path, directory_guid, file_path, file_name):
    try:
        conn = sqlite3.connect(str(db_path))
        cur = conn.cursor()
        
        # Check if file_path already exists
        cur.execute('SELECT guid FROM files WHERE file_path = ?', (file_path,))
        existing = cur.fetchone()
        
        if existing:
            return {
                'success': False,
                'error': 'File path already exists',
                'existing_guid': existing[0]
            }
        
        # File path doesn't exist, insert new record
        guid = str(uuid.uuid4())
        cur.execute(
            'INSERT INTO files (guid, directory_guid, file_path, file_name) VALUES (?, ?, ?, ?)',
            (guid, directory_guid, file_path, file_name)
        )
        conn.commit()
        conn.close()
        
        return {
            'success': True,
            'guid': guid
        }
    
    except sqlite3.IntegrityError as e:
        return {'success': False, 'error': f'Database integrity error: {str(e)}'}
    except Exception as e:
        return {'success': False, 'error': f'Database error: {str(e)}'}


def scan_directories_and_enqueue():

    print('checkpoint 1')

    extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']
    db_path='boilest.db'

    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()
    
    print('checkpoint 2')

    try:
        # Get all directories with their GUIDs
        cur.execute("SELECT guid, path FROM directories")
        directories = cur.fetchall()
        print(f'Directories found: {directories}')
    except Exception as e:
        print('Error reading directories table:', e)
        conn.close()
        return {'error': str(e)}
    finally:
        conn.close()

    files_found = 0
    files_written = 0
    files_skipped = 0
    errors = []

    for directory_guid, path in directories:
        path = os.path.expanduser(path)
        if not os.path.isdir(path):
            print(f'Directory not found: {path}')
            errors.append(f'Directory not found: {path}')
            continue
        
        # Walk directory and write each matching file to database
        for root, dirs, files in os.walk(path):
            for file in files:
                for ext in extensions:
                    if file.lower().endswith(ext.lower()):
                        files_found += 1
                        file_path = os.path.join(root, file)
                        
                        # Write file to database
                        result = write_file_to_database(db_path, directory_guid, file_path, file)
                        
                        if result['success']:
                            print(f'✓ Added: {file_path} (guid={result["guid"]})')
                            files_written += 1
                        else:
                            if 'already exists' in result.get('error', ''):
                                print(f'⊘ Skipped (exists): {file_path}')
                                files_skipped += 1
                            else:
                                print(f'✗ Error: {file_path} - {result.get("error")}')
                                errors.append(f'Failed to write {file_path}: {result.get("error")}')
                        
                        if (files_written + files_skipped) % 100 == 0:
                            print(f'Progress: {files_written} written, {files_skipped} skipped...')
                        
                        break  # Only match one extension per file
    
    summary = {
        'files_found': files_found,
        'files_written': files_written,
        'files_skipped': files_skipped,
        'errors': errors
    }
    
    print("-" * 80)
    print(f"Scan complete: {files_found} files found, {files_written} written, {files_skipped} skipped")
    if errors:
        print(f"Errors: {len(errors)}")
    return summary

result = scan_directories_and_enqueue()
print("\nResult:", result)

checkpoint 1
checkpoint 2
Directories found: [('44fb6f99-dd47-434e-9b3a-5e44c3f1fc48', '/media')]
/media\Media 1\test_file_01.mp4
Enqueued task id: 38de45eb-ba02-4d35-b1c6-9fa18424eb37
/media\Media 1\Media A\test_file_02.mp4
Enqueued task id: 9c09bb7a-8a38-4597-892e-554d84cf59d0
/media\Media 1\Media A\test_file_03.mp4
Enqueued task id: 99784c7e-08cd-4035-914e-e7bf0c020682
/media\Media 2\test_file_04.mp4
Enqueued task id: 5eca5b26-c0a0-4e88-b1fd-fc768ea9d745
/media\Media 2\test_file_05.mp4
Enqueued task id: 03e5054c-33a5-4b7e-93d3-f15c58ed461f
/media\Media 3\test_file_06.mp4
Enqueued task id: c0ac13e1-9712-4a78-a8d7-0e8036d1490c
/media\Media 4\test.mkv
Enqueued task id: 96291d32-ac63-4d10-a398-93f85f1aef0f
/media\Media 4\test_file_07.MP4
Enqueued task id: f8dc94dd-a4f6-4337-82de-df097ff544a8
Scan complete: 8 files found, 8 tasks enqueued


{'files_found': 8, 'tasks_enqueued': 8, 'errors': []}